### Import Dependecies

In [1]:
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langchain_core.messages import convert_to_openai_messages, convert_to_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import openai

import random
import ast
import inspect
import instructor
import json

from utils.utils import get_tool_descriptions, format_ai_message

from langsmith import traceable

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Retrieval Tool

In [2]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "openai", "ls_model_name": "text-embedding-3-small"}
)
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[1,1])),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def get_formatted_item_context(query: str, top_k: int = 5) -> str:

    """Get the top k context, each representing an inventory item for a given query.
    
    Args:
        query: The query to get the top k context for
        top_k: The number of context chunks to retrieve, works best with 5 or more
    
    Returns:
        A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.
    """

    context = retrieve_data(query, top_k)
    formatted_context = process_context(context)

    return formatted_context

### Structured Outputs Based Tool Calling

In [3]:
class ToolCall(BaseModel):
    name: str
    arguments: dict

class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class AgentResponse(BaseModel):
    answer: str
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")
    final_answer: bool = False
    tool_calls: List[ToolCall] = []

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    available_tools: List[Dict[str, Any]] = []

### The Prompt

In [4]:
available_tools = get_tool_descriptions([get_formatted_item_context])

In [5]:
available_tools

[{'name': 'get_formatted_item_context',
  'description': 'Get the top k context, each representing an inventory item for a given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The query to get the top k context for'},
    'top_k': {'type': 'integer',
     'description': 'The number of context chunks to retrieve, works best with 5 or more',
     'default': 5}},
   'required': ['query']},
  'returns': {'type': 'string',
   'description': 'A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.'}}]

In [6]:
initial_state = {"available_tools": available_tools}

In [7]:
prompt_template = """You are a shopping assistant that answers customer questions about products currently in stock.

## Tools

<available_tools>
{{ available_tools | tojson }}
</available_tools>

Use only the tool names listed above, exactly as written.
Place all parameters inside the "arguments" object.

## Tool Call Examples

- Search for products:
{"name": "get_formatted_item_context", "arguments": {"query": "toys for kids under 5", "top_k": 5}}

## Instructions

- Use the available tools to answer product questions. Do not fabricate product details.
- When a question involves multiple products or sub-questions, issue all tool calls at once. Never repeat a tool call you already made.
- When describing products, include detailed specifications in bullet points.
- If tools return no relevant results, tell the customer and ask them to refine their query.
- Only answer questions about products in stock. If a question is unrelated, ask the customer to clarify what product they're interested in.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Refer to retrieved data as "available products", never as "context".
- Try answering queries that are not precise, if specific names or brands are missing apply broad searches.
"""

template = Template(prompt_template)

prompt = template.render(
    available_tools=available_tools
)

In [8]:
print(prompt)

You are a shopping assistant that answers customer questions about products currently in stock.

## Tools

<available_tools>
[{"description": "Get the top k context, each representing an inventory item for a given query.", "name": "get_formatted_item_context", "parameters": {"properties": {"query": {"description": "The query to get the top k context for", "type": "string"}, "top_k": {"default": 5, "description": "The number of context chunks to retrieve, works best with 5 or more", "type": "integer"}}, "required": ["query"], "type": "object"}, "returns": {"description": "A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.", "type": "string"}}]
</available_tools>

Use only the tool names listed above, exactly as written.
Place all parameters inside the "arguments" object.

## Tool Call Examples

- Search for products:
{"name": "get_formatted_item_context", "arguments": {"query": "toys for kids un

### Agent Node

In [9]:
@traceable(
    name="agent_node",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-4.1-mini"}
)
def agent_node(state) -> dict:

    prompt_template = """You are a shopping assistant that answers customer questions about products currently in stock.

## Tools

<available_tools>
{{ available_tools | tojson }}
</available_tools>

Use only the tool names listed above, exactly as written.
Place all parameters inside the "arguments" object.

## Tool Call Examples

- Search for products:
{"name": "get_formatted_item_context", "arguments": {"query": "toys for kids under 5", "top_k": 5}}

## Instructions

- Use the available tools to answer product questions. Do not fabricate product details.
- When a question involves multiple products or sub-questions, issue all tool calls at once. Never repeat a tool call you already made.
- When describing products, include detailed specifications in bullet points.
- If tools return no relevant results, tell the customer and ask them to refine their query.
- Only answer questions about products in stock. If a question is unrelated, ask the customer to clarify what product they're interested in.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Refer to retrieved data as "available products", never as "context".
- Try answering queries that are not precise, if specific names or brands are missing apply broad searches.
"""

    template = Template(prompt_template)

    prompt = template.render(
        available_tools=state.available_tools
    )

    messages = state.messages

    conversation = []

    for message in messages:
        conversation.append(convert_to_openai_messages(message))

    client = instructor.from_provider(
        "openai/gpt-4.1-mini"
    )

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt},
            *conversation
        ],
        response_model=AgentResponse
    )

    ai_message = format_ai_message(response)

    
    return {
        "messages": [ai_message],
        "tool_calls": response.tool_calls,
        "answer": response.answer,
        "final_answer": response.final_answer,
        "references": response.references
    }

In [10]:
initial_state = State(
    messages=[{"role": "user", "content": "Can I get a tablet for my kid, a watch for me a laptop for my wife and a waterproof speaker for our party next week?"}],
    available_tools=available_tools
)

In [11]:
result = agent_node(initial_state)

In [12]:
result

{'messages': [AIMessage(content='I will look for a tablet suitable for kids, a watch for you, a laptop for your wife, and a waterproof speaker for your party next week from our available products.', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'get_formatted_item_context', 'args': {'query': 'tablet for kids', 'top_k': 2}, 'id': 'call_0', 'type': 'tool_call'}, {'name': 'get_formatted_item_context', 'args': {'query': 'watch', 'top_k': 2}, 'id': 'call_1', 'type': 'tool_call'}, {'name': 'get_formatted_item_context', 'args': {'query': 'laptop', 'top_k': 2}, 'id': 'call_2', 'type': 'tool_call'}, {'name': 'get_formatted_item_context', 'args': {'query': 'waterproof speaker', 'top_k': 2}, 'id': 'call_3', 'type': 'tool_call'}], invalid_tool_calls=[])],
 'tool_calls': [ToolCall(name='get_formatted_item_context', arguments={'query': 'tablet for kids', 'top_k': 2}),
  ToolCall(name='get_formatted_item_context', arguments={'query': 'watch', 'top_k': 2}),
  ToolCall(name='get_form

### Exploring Native OpenAI Tool Calling

In [13]:
from openai import OpenAI

In [14]:
client = OpenAI()

In [15]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": """You are a shopping assistant that answers customer questions about products currently in stock.

## Instructions

- Use the available tools to answer product questions. Do not fabricate product details.
- When a question involves multiple products or sub-questions, issue all tool calls at once. Never repeat a tool call you already made.
- When describing products, include detailed specifications in bullet points.
- If tools return no relevant results, tell the customer and ask them to refine their query.
- Only answer questions about products in stock. If a question is unrelated, ask the customer to clarify what product they're interested in.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Refer to retrieved data as "available products", never as "context".
- Try answering queries that are not precise, if specific names or brands are missing apply broad searches.
"""},
        {"role": "user", "content": "Can I get a tablet for my kid, a watch for me a laptop for my wife and a waterproof speaker for our party next week?"}
    ],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_formatted_item_context",
                "description": "Get the top k context, each representing an inventory item for a given query",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The query to get the top k context for"},
                        "top_k": {"type": "integer", "description": "The number of context chunks to retrieve, works best with 5 or more"}
                    },
                    "required": ["query"]
                }
            }
        }
    ],
    tool_choice="required"
)

In [16]:
response

ChatCompletion(id='chatcmpl-DcZlU6I2zcTYhoRR0Ntf3ChmfMWIc', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_sGko0Nie8PuLJEEmFZpCxPzB', function=Function(arguments='{"query": "tablet for kids", "top_k": 3}', name='get_formatted_item_context'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_t4or4BZepzlqsn1m3nQS0Epi', function=Function(arguments='{"query": "smartwatch", "top_k": 3}', name='get_formatted_item_context'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_9k9TYwckwo3LiQhQqNHdcZYd', function=Function(arguments='{"query": "laptop", "top_k": 3}', name='get_formatted_item_context'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_FzBCfT1N6VB0n6TfpsJZ9J5P', function=Function(arguments='{"query": "waterproof speaker", "top_k": 3}', n

### Exploring LangChain Tool Calling

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [18]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "openai", "ls_model_name": "text-embedding-3-small"}
)
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[1,1])),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


@tool
def get_formatted_item_context(query: str, top_k: int = 5) -> str:

    """Get the top k context, each representing an inventory item for a given query.
    
    Args:
        query: The query to get the top k context for
        top_k: The number of context chunks to retrieve, works best with 5 or more
    
    Returns:
        A string of the top k context chunks with IDs and average ratings prepending each chunk, each representing an inventory item for a given query.
    """

    context = retrieve_data(query, top_k)
    formatted_context = process_context(context)

    return formatted_context

In [19]:
llm = ChatOpenAI(model="gpt-4.1-mini")

In [20]:
llm_with_tools = llm.bind_tools(
    [get_formatted_item_context],
    tool_choice="auto"
)

In [21]:
from langchain_core.messages import HumanMessage, SystemMessage

In [22]:
prompt_template = """You are a shopping assistant that answers customer questions about products currently in stock.

## Instructions

- Use the available tools to answer product questions. Do not fabricate product details.
- When a question involves multiple products or sub-questions, issue all tool calls at once. Never repeat a tool call you already made.
- When describing products, include detailed specifications in bullet points.
- If tools return no relevant results, tell the customer and ask them to refine their query.
- Only answer questions about products in stock. If a question is unrelated, ask the customer to clarify what product they're interested in.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Refer to retrieved data as "available products", never as "context".
- Try answering queries that are not precise, if specific names or brands are missing apply broad searches.
"""

template = Template(prompt_template)

prompt = template.render()

messages = [HumanMessage(content="Can I get a tablet for my kid, a watch for me a laptop for my wife and a waterproof speaker for our party next week?")]

response = llm_with_tools.invoke(
    [
        SystemMessage(content=prompt),
        *messages
    ]
)

In [23]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 335, 'total_tokens': 449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_49f260aaf7', 'id': 'chatcmpl-DcZlxsg8QHb0401r9o35yzTFdHrXU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dfe34-6d35-7ca3-8549-8a13856174ad-0', tool_calls=[{'name': 'get_formatted_item_context', 'args': {'query': 'tablet for kids', 'top_k': 3}, 'id': 'call_vxk2KCzdmy2c1zrTFROPCnGV', 'type': 'tool_call'}, {'name': 'get_formatted_item_context', 'args': {'query': 'smartwatch for adults', 'top_k': 3}, 'id': 'call_FhsDwlO2qRvjQMNk4kLWbDye', 'type': 'tool_call'}, {'name': 'get_formatted_ite